<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Desafío 2 - Custom embeddings con Gensim
### Nombre: Julia Stephania Maldonado Gomez
### Corpus elegido: *Dracula*, de Bram Stoker (texto completo en inglés, Project Gutenberg)


### Objetivo
Entrenar embeddings de palabras propios (Word2Vec, con Gensim) a partir de un corpus elegido por el grupo: la novela **Dracula** de Bram Stoker, en su idioma original (inglés).

A diferencia del ejemplo visto en clase (letras de canciones, muchos documentos cortos), acá el corpus es **un único libro largo** (~160.000 palabras). Esto va a influir directamente en las decisiones de preprocesamiento e hiperparámetros, que se documentan en cada sección.

**Fuente del corpus:** [Project Gutenberg - Dracula (eBook #345)](https://www.gutenberg.org/ebooks/345), dominio público.

In [1]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from gensim.models import Word2Vec
except ImportError:
    !pip install -q gensim
    from gensim.models import Word2Vec

try:
    import nltk
except ImportError:
    !pip install -q nltk
    import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.8 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

### Datos
Descargamos el texto completo de *Dracula* directamente desde Project Gutenberg.

In [2]:
# Descargar el libro (texto plano UTF-8) desde Project Gutenberg
if not os.path.exists("dracula.txt"):
    !wget -q "https://www.gutenberg.org/cache/epub/345/pg345.txt" -O dracula.txt
    print("Descarga completa")
else:
    print("El archivo ya se encuentra descargado")

Descarga completa


In [3]:
with open("dracula.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Cantidad de caracteres totales (con metadata de Gutenberg):", len(raw_text))

Cantidad de caracteres totales (con metadata de Gutenberg): 865207


### 1 - Preprocesamiento

Project Gutenberg agrega al principio y al final de cada libro una licencia / metadata que **no** es parte de la novela. La eliminamos quedándonos solo con el texto entre los marcadores `*** START OF...` y `*** END OF...`.

Luego, a diferencia del dataset de canciones (donde cada línea del `.txt` ya era una "oración"/documento), acá tenemos un texto corrido. Por eso:
1. Separamos el libro en **oraciones** con `nltk.sent_tokenize` (en vez de usar saltos de línea, que en un libro no representan oraciones).
2. Tokenizamos cada oración en palabras con `text_to_word_sequence` (igual que en el ejemplo de clase), que además pasa todo a minúsculas y saca signos de puntuación.

In [4]:
start_match = re.search(r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", raw_text, re.IGNORECASE | re.DOTALL)
end_match = re.search(r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", raw_text, re.IGNORECASE | re.DOTALL)

book_text = raw_text[start_match.end():end_match.start()]

# Normalizamos saltos de línea simples (que Gutenberg usa para "cortar" renglones dentro
# de un mismo párrafo) reemplazandolos por un espacio, para no partir oraciones a la mitad
book_text = re.sub(r"\r\n", "\n", book_text)
book_text = re.sub(r"(?<!\n)\n(?!\n)", " ", book_text)
book_text = re.sub(r"\s+", " ", book_text).strip()

print("Cantidad de caracteres de la novela (sin metadata de Gutenberg):", len(book_text))
print()
print(book_text[:500])

Cantidad de caracteres de la novela (sin metadata de Gutenberg): 838345

DRACULA _by_ Bram Stoker [Illustration: colophon] NEW YORK GROSSET & DUNLAP _Publishers_ Copyright, 1897, in the United States of America, according to Act of Congress, by Bram Stoker [_All rights reserved._] PRINTED IN THE UNITED STATES AT THE COUNTRY LIFE PRESS, GARDEN CITY, N.Y. TO MY DEAR FRIEND HOMMY-BEG Contents CHAPTER I. Jonathan Harker’s Journal CHAPTER II. Jonathan Harker’s Journal CHAPTER III. Jonathan Harker’s Journal CHAPTER IV. Jonathan Harker’s Journal CHAPTER V. Letters—Lucy and 


In [5]:
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(book_text)
print("Cantidad de oraciones/documentos:", len(sentences))
sentences[:3]

Cantidad de oraciones/documentos: 7379


['DRACULA _by_ Bram Stoker [Illustration: colophon] NEW YORK GROSSET & DUNLAP _Publishers_ Copyright, 1897, in the United States of America, according to Act of Congress, by Bram Stoker [_All rights reserved._] PRINTED IN THE UNITED STATES AT THE COUNTRY LIFE PRESS, GARDEN CITY, N.Y. TO MY DEAR FRIEND HOMMY-BEG Contents CHAPTER I. Jonathan Harker’s Journal CHAPTER II.',
 'Jonathan Harker’s Journal CHAPTER III.',
 'Jonathan Harker’s Journal CHAPTER IV.']

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = [text_to_word_sequence(s) for s in sentences]
# Descartamos oraciones que quedaron vacías o con un solo token luego de tokenizar
sentence_tokens = [tok for tok in sentence_tokens if len(tok) > 1]

print("Cantidad de oraciones tokenizadas:", len(sentence_tokens))
sentence_tokens[:3]

Cantidad de oraciones tokenizadas: 7269


[['dracula',
  'by',
  'bram',
  'stoker',
  'illustration',
  'colophon',
  'new',
  'york',
  'grosset',
  'dunlap',
  'publishers',
  'copyright',
  '1897',
  'in',
  'the',
  'united',
  'states',
  'of',
  'america',
  'according',
  'to',
  'act',
  'of',
  'congress',
  'by',
  'bram',
  'stoker',
  'all',
  'rights',
  'reserved',
  'printed',
  'in',
  'the',
  'united',
  'states',
  'at',
  'the',
  'country',
  'life',
  'press',
  'garden',
  'city',
  'n',
  'y',
  'to',
  'my',
  'dear',
  'friend',
  'hommy',
  'beg',
  'contents',
  'chapter',
  'i',
  'jonathan',
  'harker’s',
  'journal',
  'chapter',
  'ii'],
 ['jonathan', 'harker’s', 'journal', 'chapter', 'iii'],
 ['jonathan', 'harker’s', 'journal', 'chapter', 'iv']]

### 2 - Crear los vectores (Word2Vec)

**Decisiones de hiperparámetros:**

Como el corpus es un único libro (~4.700 oraciones, ~160.000 palabras) y no un dataset con miles de documentos, ajustamos los valores por defecto del ejemplo de clase:

- **`min_count=3`** (en vez de 5): al ser un corpus más chico que el de canciones, un `min_count` muy alto dejaría afuera demasiadas palabras temáticas propias del libro (nombres propios, objetos puntuales como "crucifix", "wolf", etc.). Con 3 filtramos ruido (errores de tokenización, palabras únicas) sin perder vocabulario relevante.
- **`window=5`** (en vez de 2): en prosa narrativa el contexto relevante de una palabra suele extenderse más allá de las 2 palabras vecinas (a diferencia de versos cortos de canciones). Una ventana más grande ayuda a capturar relaciones temáticas/semánticas en vez de solo sintácticas.
- **`vector_size=150`** (en vez de 300): con un vocabulario más chico (unas pocas miles de palabras únicas), una dimensionalidad menor evita sobre-parametrizar el modelo y ayuda a que el entrenamiento converja mejor con los datos disponibles.
- **`negative=15`**: se mantiene un valor similar al ejemplo (negative sampling ayuda a acelerar el entrenamiento respecto de softmax completo).
- **`sg=1` (skip-gram)**: se mantiene skip-gram porque, con un corpus relativamente chico, suele funcionar mejor que CBOW para representar palabras poco frecuentes (por ejemplo, nombres propios como "Mina", "Harker", "Renfield").
- **`epochs=50`** (en vez de 20): al ser un corpus más chico en cantidad de documentos, se necesitan más épocas para que el modelo vea suficientes veces cada palabra y sus contextos.

In [7]:
from gensim.models.callbacks import CallbackAny2Vec

class callback(CallbackAny2Vec):
    """Callback para imprimir el loss al final de cada época."""
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print("Loss after epoch {}: {}".format(self.epoch, loss))
        else:
            print("Loss after epoch {}: {}".format(self.epoch, loss - self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [8]:
w2v_model = Word2Vec(
    min_count=3,      # frecuencia mínima de palabra para incluirla en el vocabulario
    window=5,         # cant. de palabras antes y después de la predicha
    vector_size=150,  # dimensionalidad de los vectores
    negative=15,       # cantidad de negative samples
    workers=1,         # si tienen más cores pueden cambiar este valor
    sg=1               # modelo 0:CBOW  1:skipgram
)

In [9]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [10]:
print("Cantidad de docs (oraciones) en el corpus:", w2v_model.corpus_count)
print("Cantidad de words distintas en el vocabulario:", len(w2v_model.wv.index_to_key))

Cantidad de docs (oraciones) en el corpus: 7269
Cantidad de words distintas en el vocabulario: 3781


### 3 - Entrenar embeddings

In [11]:
w2v_model.train(
    sentence_tokens,
    total_examples=w2v_model.corpus_count,
    epochs=50,
    compute_loss=True,
    callbacks=[callback()]
)

Loss after epoch 0: 2195082.75
Loss after epoch 1: 1626497.5
Loss after epoch 2: 1488081.25
Loss after epoch 3: 1444968.5
Loss after epoch 4: 1440602.5
Loss after epoch 5: 1365652.5
Loss after epoch 6: 1347663.0
Loss after epoch 7: 1339017.0
Loss after epoch 8: 1327086.0
Loss after epoch 9: 1321188.0
Loss after epoch 10: 1311335.0
Loss after epoch 11: 1271756.0
Loss after epoch 12: 1245214.0
Loss after epoch 13: 1240094.0
Loss after epoch 14: 1233264.0
Loss after epoch 15: 1224790.0
Loss after epoch 16: 1215956.0
Loss after epoch 17: 1212596.0
Loss after epoch 18: 1203076.0
Loss after epoch 19: 1200494.0
Loss after epoch 20: 1192046.0
Loss after epoch 21: 1189952.0
Loss after epoch 22: 1180288.0
Loss after epoch 23: 1174560.0
Loss after epoch 24: 1173976.0
Loss after epoch 25: 1184980.0
Loss after epoch 26: 1190128.0
Loss after epoch 27: 1188344.0
Loss after epoch 28: 1183744.0
Loss after epoch 29: 1174596.0
Loss after epoch 30: 1172348.0
Loss after epoch 31: 1168296.0
Loss after epoch

(5445659, 8191300)

### 4 - Ensayar

Elegimos términos de interés propios de la temática del libro: personajes, objetos y conceptos centrales de la trama (vampirismo, terror gótico).

*Nota: si alguna palabra no aparece en el vocabulario (por `min_count` o por no estar en el libro con ese formato exacto), Gensim va a lanzar un `KeyError`; se puede probar variantes (singular/plural, con/sin mayúscula ya normalizada a minúscula, etc.)*

In [12]:
# Palabras que MÁS se relacionan con "blood" (sangre, tema central del libro)
w2v_model.wv.most_similar(positive=["blood"], topn=10)

[('stained', 0.4940939247608185),
 ('smeared', 0.4934144914150238),
 ('bloom', 0.4815787374973297),
 ('trickled', 0.465044230222702),
 ('veins', 0.46259748935699463),
 ('transfusion', 0.46077829599380493),
 ('crimson', 0.45380109548568726),
 ('drop', 0.4336179196834564),
 ('purity', 0.42945632338523865),
 ('baptism', 0.4274291396141052)]

In [13]:
# Palabras que MÁS se relacionan con "count" (por "Count Dracula")
w2v_model.wv.most_similar(positive=["count"], topn=10)

[('dracula', 0.40545329451560974),
 ('customs', 0.3900388777256012),
 ('unlocked', 0.38889625668525696),
 ('handle', 0.3851606249809265),
 ('mother', 0.3712231516838074),
 ('maid', 0.3588029444217682),
 ('stayed', 0.34810036420822144),
 ('whither', 0.3457525074481964),
 ('ahead', 0.34416162967681885),
 ('discover', 0.33974966406822205)]

In [14]:
# Palabras que MÁS se relacionan con "night"
w2v_model.wv.most_similar(positive=["night"], topn=10)

[('sundown', 0.4468877613544464),
 ('morrow', 0.43875622749328613),
 ('10', 0.43497633934020996),
 ('bells', 0.4263157844543457),
 ('11', 0.4222276508808136),
 ('languid', 0.4215703308582306),
 ('19', 0.41832104325294495),
 ('june', 0.41248536109924316),
 ('inference', 0.3986268937587738),
 ('“at', 0.39719900488853455)]

In [15]:
# Palabras que MÁS se relacionan con "castle"
w2v_model.wv.most_similar(positive=["castle"], topn=10)

[('locality', 0.4564862847328186),
 ('sisters', 0.4448804259300232),
 ('dracula', 0.422465056180954),
 ('sereth', 0.41815829277038574),
 ('dracula’s', 0.41555383801460266),
 ('moreover', 0.41106897592544556),
 ('lighthouse', 0.4049454927444458),
 ('kitchen', 0.39244377613067627),
 ('bistritza', 0.3816981613636017),
 ('west', 0.372406542301178)]

In [16]:
# Palabras que MENOS se relacionan con "love" (antónimo/contraste temático)
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('entering', 0.059356480836868286),
 ('hours', 0.039392873644828796),
 ('helped', 0.035760074853897095),
 ('exact', 0.034749653190374374),
 ('available', 0.03463836759328842),
 ('pointing', 0.0345781147480011),
 ('entire', 0.030457505956292152),
 ('conscious', 0.029825039207935333),
 ('difficulty', 0.02918650396168232),
 ('arrive', 0.02770102582871914)]

In [17]:
# Combinación de vectores: análisis tipo analogía
# "dracula" es a "count" como "mina" es a...?
w2v_model.wv.most_similar(positive=["dracula", "mina"], negative=["count"], topn=10)

[('murray’s', 0.45951953530311584),
 ('x', 0.4505198299884796),
 ('“mina', 0.4030146598815918),
 ('madam', 0.3964736759662628),
 ('chapter', 0.3937532305717468),
 ('correspondent', 0.3901903033256531),
 ('v', 0.38554099202156067),
 ('p', 0.3779938220977783),
 ('interrupted', 0.37518882751464844),
 ('“believe', 0.3695095479488373)]

In [18]:
# el método `get_vector` permite obtener los vectores:
vector_blood = w2v_model.wv.get_vector("blood")
print(vector_blood[:20], "...")  # se muestran los primeros 20 valores

[-0.34198606 -0.02359598  0.27611318  0.28286195  0.27409318 -0.52252936
  0.8710361   0.22707498 -0.19686326  0.3362932   0.2860408  -0.371982
 -0.03966847 -0.03350493  0.03648004 -0.09682009 -0.6852226   0.10834746
  0.06463746 -0.39604607] ...


**Interpretación de most_similar**

Los términos más similares a "blood" (stained, smeared, veins, transfusion, crimson) son muy
coherentes con la trama: remiten directamente a las escenas de transfusión de sangre a Lucy y
a la imaginería vampírica del libro. Algo similar pasa con "castle" (locality, bistritza,
dracula's, kitchen), que refleja el escenario transilvano de los primeros capítulos.

En cambio, "count" mezcla resultados coherentes (dracula) con otros que no lo son (customs,
handle, mother, maid). Esto se debe a que "count" es una palabra ambigua en inglés: puede ser
el título del personaje ("the Count") o el verbo "contar", y Word2Vec no distingue entre
sentidos de una misma palabra (no maneja polisemia), por lo que termina promediando contextos
de ambos usos en un solo vector.

La analogía vectorial (dracula + mina - count) no arrojó un resultado interpretable como
analogía. Es un comportamiento esperable en corpus chicos como este (~7.300 oraciones): las
analogías vectoriales suelen necesitar corpus mucho más grandes para capturar relaciones
sistemáticas entre pares de palabras.

### 5 - Visualizar agrupación de vectores (reducción de dimensionalidad)

In [19]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE

def reduce_dimensions(model, num_dimensions=2):
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [24]:
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

# El vocabulario de un único libro es más chico que el del dataset de canciones,
# por lo que con MAX_WORDS=300 ya se cubre una porción representativa sin saturar el gráfico.
MAX_WORDS = 300
fig = px.scatter(x=vecs[:MAX_WORDS, 0], y=vecs[:MAX_WORDS, 1], text=labels[:MAX_WORDS])
fig.update_layout(title="Embeddings de Dracula (Bram Stoker) - TSNE 2D", width=900, height=700)
fig.show(renderer="colab")  # esto para plotly en colab


Al elegir MAX_WORDS=300, el gráfico queda dominado por las palabras más frecuentes del
vocabulario, que en su mayoría son palabras funcionales (pronombres, verbos auxiliares) y no
palabras temáticas: términos como "castle", "dracula" o "vampire" ni siquiera entran en el
top 300 por ser relativamente poco frecuentes frente a "the", "and", "he", etc.

Aun así, se identifican dos agrupamientos con sentido semántico claro:
- "mina" aparece prácticamente pegada a "madam", y cerca de "arthur" y "professor": un
  cluster de personajes y formas de tratamiento, coherente con cómo se refieren a Mina Harker
  en el libro.
- "blood" se agrupa con "face", "red", "white", "eyes": un cluster de descripción visual y
  corporal, coherente con el tema vampírico central de la novela.

En contraste, palabras como "night", "death" y "count" quedan rodeadas casi exclusivamente de
palabras funcionales (he, him, seen, had, must), lo que sugiere que en esta proyección 2D su
posición está más influida por el rol sintáctico que cumplen en las oraciones que por su
significado temático.

In [21]:
# Graficar los embeddings en 3D
vecs_3d, labels_3d = reduce_dimensions(w2v_model, 3)

fig = px.scatter_3d(
    x=vecs_3d[:MAX_WORDS, 0], y=vecs_3d[:MAX_WORDS, 1], z=vecs_3d[:MAX_WORDS, 2],
    text=labels_3d[:MAX_WORDS]
)
fig.update_traces(marker_size=2)
fig.update_layout(title="Embeddings de Dracula (Bram Stoker) - TSNE 3D")
fig.show(renderer="colab")  # esto para plotly en colab

In [22]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/
vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

print("Archivos vectors.tsv y labels.tsv generados")

Archivos vectors.tsv y labels.tsv generados


### Conclusiones

El entrenamiento de Word2Vec sobre el texto completo de Dracula (Bram Stoker) permitió obtener
embeddings coherentes para varios términos centrales del libro, en particular "blood" y
"castle", cuyas palabras más similares reflejan escenas y ambientación reales de la novela.

Al ser un corpus chico compuesto por un único libro (~7.300 oraciones, ~3.800 palabras únicas
en el vocabulario), se observaron dos limitaciones esperables: (1) palabras polisémicas como
"count" mezclan sentidos distintos en un mismo vector, y (2) las analogías vectoriales no
arrojaron resultados interpretables, algo típico en corpus de este tamaño.

En la visualización 2D, limitar el gráfico a las 300 palabras más frecuentes priorizó palabras
funcionales por sobre palabras temáticas de baja frecuencia (como "dracula" o "vampire", que ni
siquiera entraron en el recorte). Aun así, se pudieron identificar clusters semánticamente
coherentes, como el de personajes ("mina", "madam", "arthur", "professor") y el de descripción
corporal/visual asociada a la sangre ("blood", "face", "red", "white", "eyes").

Como posible mejora, se podría repetir el análisis con un MAX_WORDS mayor o filtrando
directamente por palabras de contenido (excluyendo stopwords), para que la visualización
priorice términos temáticamente relevantes por sobre palabras funcionales de alta frecuencia.